# tps-meter — Colab çalıştırıcı

Kod Drive'daki zip paketinden gelir; **iş Colab'ın yerel diskinde** yapılır (hızlı), bitmiş her parça anında **Drive'a yedeklenir**. Oturum kapanırsa 1–2. hücreleri ve üretim hücresini yeniden çalıştırmak yeter: kaldığı yerden devam eder.

**Drive düzeni** (`Drive'ım/tps-meter/`):
- `tps-meter-code.zip` — bilgisayarda `python scripts/pack_for_colab.py` ile üretilen kod paketi
- `data/synthetic/v1/` — üretilen veri ve yedek parçalar (defter yazar)
- `runs/` — eğitim çıktıları (Faz 5)

**Çalışma zamanı:** Veri üretimi yalnızca CPU kullanır ve bellek bant genişliğine bağlıdır; çok çekirdekli bir çalışma zamanı seçin (*Çalışma zamanı → Çalışma zamanı türünü değiştir*). 2. hücre çekirdek sayısını yazar, 4. hücre en iyi işçi sayısını ölçer.

In [ ]:
# 1) Drive + kod + yollar
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, shutil, zipfile
DRIVE_DIR = pathlib.Path('/content/drive/MyDrive/tps-meter')
WORK = pathlib.Path('/content/tps-meter')
if WORK.exists():
    shutil.rmtree(WORK)
zipfile.ZipFile(DRIVE_DIR / 'tps-meter-code.zip').extractall(WORK)

LOCAL_DATA = pathlib.Path('/content/data')              # hızlı yerel disk
MIRROR = DRIVE_DIR / 'data' / 'synthetic' / 'v1'          # kalıcı yedek
LOCAL_DATA.mkdir(exist_ok=True)
MIRROR.mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / 'runs').mkdir(parents=True, exist_ok=True)
os.environ['TPS_DATA_ROOT'] = str(LOCAL_DATA)
os.environ['TPS_RUNS_ROOT'] = str(DRIVE_DIR / 'runs')
os.chdir(WORK)
print('kod:', WORK, '| yerel veri:', LOCAL_DATA, '| Drive yedeği:', MIRROR)
print('Drive yedeğinde:', sorted(p.name for p in MIRROR.iterdir()))

In [ ]:
# 2) Kurulum ve donanım
!pip install -q -e ".[dev]"
# The running kernel does not see a package installed in the same session; add it to the path.
import os, sys
sys.path.insert(0, str(WORK / 'src'))
import torch
print('CPU çekirdeği:', os.cpu_count())
!free -g | head -2
from tpslib.runtime import get_device, describe_device
print('torch', torch.__version__, '|', describe_device(get_device()))

In [ ]:
# 3) Testler (yaklaşık 1–2 dk). Hepsi geçmeden üretime başlamayın.
!python -m pytest -q

In [ ]:
# 4) Verim testi: en hızlı işçi sayısını bulur ve tam üretim süresini tahmin eder (birkaç dakika).
!python scripts/bench_generation.py

In [ ]:
# 5) TAM ÜRETİM. WORKERS'ı 4. hücrenin önerdiği değerle değiştirin.
# Oturum koparsa: 1, 2 ve bu hücreyi yeniden çalıştırın; bitmiş parçalar Drive'dan geri yüklenir.
WORKERS = os.cpu_count()
!python scripts/generate_dataset.py --splits train val test test_interp test_faulty --workers {WORKERS} --mirror "{MIRROR}"

In [ ]:
# 6) Kontrol: Drive'daki dosyalar tam mı?
import h5py, json
for name in ['train', 'val', 'test', 'test_interp', 'test_faulty']:
    f = MIRROR / f'{name}.h5'
    if not f.exists():
        print(f'{name:12s} YOK'); continue
    with h5py.File(f, 'r') as h:
        st = json.loads(h.attrs['stats'])
        print(f"{name:12s} n={h['dT'].shape[0]:6d} ok={st['n_ok']:6d} hata={st['n_failed']} "
              f"ort. sim={st['sim_time_mean_s']:.1f}s  {f.stat().st_size/1e6:.0f} MB")
print(json.dumps(json.loads((MIRROR / 'manifest.json').read_text())['splits'], indent=1)[:2000])

## Faz 5 — M2 eğitimi

**Çalışma zamanı:** GPU (L4 yeterli; A100 da olur). Eğitim verisi her epokta CPU'da yeniden gürültülendiği için çekirdek sayısı da önemli (L4: 12).

Oturum koparsa: 1, 2, 7. hücreleri ve kaldığınız eğitim hücresini **aynı komutla** yeniden çalıştırın; koşular Drive'daki `runs/` altında son epoktan devam eder.

Sırayla: 8 (5 epok deneme) → 9 (veri miktarı çalışması) → 10 (5 tohumlu topluluk) → 11 (ablasyon).

In [ ]:
# 7) Veriyi Drive'dan hızlı yerel diske kopyala (eğitim buradan okur)
dst = LOCAL_DATA / 'synthetic' / 'v1'
dst.mkdir(parents=True, exist_ok=True)
for name in ['train', 'val', 'test', 'test_interp', 'test_faulty']:
    src = MIRROR / f'{name}.h5'
    if src.exists() and not (dst / src.name).exists():
        shutil.copy2(src, dst / src.name)
shutil.copy2(MIRROR / 'manifest.json', dst / 'manifest.json')
print(sorted(p.name for p in dst.iterdir()))
NW = os.cpu_count()

In [ ]:
# 8) Kısa deneme: tek tohum, 5 epok (süre ve öğrenme eğrisi kontrolü)
!python scripts/train_m2.py --seeds 1 --max-epochs 5 --tag trial --num-workers {NW}

In [ ]:
# 9) Veri miktarı çalışması: 10k ve 25k örnekle (50k = 10. hücredeki seed1). Daha fazla veri gerekir mi sorusunun cevabı.
!python scripts/train_m2.py --seeds 1 --max-train 10000 --tag n10k --num-workers {NW}
!python scripts/train_m2.py --seeds 1 --max-train 25000 --tag n25k --num-workers {NW}

In [ ]:
# 9b) Teşhis: hatanın hangi örneklerden geldiği (yalnızca val; test setleri kilitli kalır). ~2 dk
!python scripts/diagnose.py --run n10k_m2_seed1 --num-workers {NW}


In [ ]:
# 9c) Karşılaştırma (tek tohum, 50k): (a) tüm veri, (b) temas asimetrisi <= 10. Her biri ~40 dk.
# Sonra her ikisinin teşhisi. Bitince son iki tabloyu gönderin.
!python scripts/train_m2.py --seeds 1 --tag v1 --num-workers {NW}
!python scripts/train_m2.py --seeds 1 --tag v1sym --max-asym 10 --num-workers {NW}
!python scripts/diagnose.py --run v1_m2_seed1 --num-workers {NW}
!python scripts/diagnose.py --run v1sym_m2_seed1 --num-workers {NW}


In [ ]:
# 9d) (a) modelini tamamla: erken durdurma artık 30. epoktan önce devreye girmiyor.
# v1_m2_seed1 28. epoktan kaldığı yerden 60'a devam eder (~23 dk), sonra teşhis.
!python scripts/train_m2.py --seeds 1 --tag v1 --num-workers {NW}
!python scripts/diagnose.py --run v1_m2_seed1 --num-workers {NW}


In [ ]:
# 10) Tam eğitim: 5 tohumlu topluluk, gerçekçi temas (asimetri <= 10; seed1 = 9c'deki v1sym)
!python scripts/train_m2.py --seeds 1 2 3 4 5 --tag v1sym --max-asym 10 --num-workers {NW}


In [ ]:
# 11) Ablasyon: k = 1,5–2,5 W/m·K bandı train/val'dan çıkarılmış model (tek tohum, asimetri <= 10)
!python scripts/train_m2.py --seeds 1 --tag ablation --max-asym 10 --exclude-k-band 1.5 2.5 --num-workers {NW}


In [ ]:
# 12) Özet: her koşunun en iyi epoğu
import json, glob
for f in sorted(glob.glob(str(DRIVE_DIR / 'runs' / '*' / 'summary.json'))):
    s = json.load(open(f)); b = s['best']
    print(f"{pathlib.Path(f).parent.name:28s} ep={s['best_epoch']:3d}/{s['epochs_run']:3d} "
          f"k med={100*b['val_med_rel_err_ln_k']:.2f}% p95={100*b['val_p95_rel_err_ln_k']:.2f}% "
          f"a med={100*b['val_med_rel_err_ln_alpha']:.2f}% cov95(k)={b['val_cov95_ln_k']:.2f}")

## Faz 6 — Değerlendirme, kompakt model, ONNX

Test setleri burada **bir kez** kullanılır. Çıktılar Drive'da `tps-meter/reports/phase6/` ve `tps-meter/phase6_bundle.zip`.

In [ ]:
# 13) Kompakt cihaz modeli (~62k parametre, 2 üye). Topluluk eğitiminden bağımsız; ayrı bir oturumda paralel de çalışabilir. ~1 saat
!python scripts/train_m2.py --config configs/train_m2_compact.yaml --seeds 1 2 --tag v1c --max-asym 10 --num-workers {NW}

In [ ]:
# 14) Son değerlendirme (topluluk + ablasyon + kompakt hazır olmalı). ~20–30 dk
!python scripts/evaluate.py --ensemble-tag v1sym --seeds 1 2 3 4 5 --ablation ablation_m2_seed1 --compact-tag v1c --compact-seeds 1 2 --max-asym 10 --num-workers {NW}

## Güven kontrolleri v2 (Faz 6 sonrası)

Model yeniden eğitilmez. Eşikler ve varyans ölçeği **val**'dan seçilir. Bu koşu test setlerinin **ikinci kullanımıdır**; çıktılar `tps-meter/reports/phase6_trust/` ve `tps-meter/phase6_trust_bundle.zip`. Hücre tek başına çalışır (1–2. hücreler gerekmez).


In [ ]:
# 15) Güven kontrolleri v2 + varyans ölçeği (TEST SETLERİNİN 2. KULLANIMI). Tek başına çalışır; ~10 dk.
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, shutil, zipfile
DRIVE_DIR = pathlib.Path('/content/drive/MyDrive/tps-meter')
WORK = pathlib.Path('/content/tps-meter')
shutil.rmtree(WORK, ignore_errors=True)
zipfile.ZipFile(DRIVE_DIR / 'tps-meter-code.zip').extractall(WORK)
os.chdir(WORK)
dst = pathlib.Path('/content/data/synthetic/v1'); dst.mkdir(parents=True, exist_ok=True)
for f in ['val.h5', 'test.h5', 'test_faulty.h5', 'manifest.json']:
    if not (dst / f).exists():
        shutil.copy2(DRIVE_DIR / 'data' / 'synthetic' / 'v1' / f, dst / f)
os.environ['TPS_DATA_ROOT'] = '/content/data'
os.environ['TPS_RUNS_ROOT'] = str(DRIVE_DIR / 'runs')
!pip install -q -e ".[dev]"
!python -m pytest -q tests/test_trust.py
!python scripts/evaluate.py --trust-only --ensemble-tag v1sym --seeds 1 2 3 4 5 --max-asym 10 --num-workers {os.cpu_count()}
print((DRIVE_DIR / 'reports' / 'phase6_trust' / 'TRUST_REPORT.md').read_text(encoding='utf-8'))
